# Attribute Distributions:

----

# Import Libraries:

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F 
from pyspark.sql.functions import * 
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, log, explode, expr
import numpy as np
import pandas as pd 
import os

In [ ]:
# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("correlation_attributes_distribution")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .config("spark.network.timeout", "600s")
    .config("spark.driver.maxResultSize", "4g")
    .config("spark.rpc.askTimeout", "600s")
    .config("spark.driver.memory", "100G")
    .config("spark.executor.memory", "100G")
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "Etc/UTC")
    .config("spark.sql.debug.maxToStringFields", "1000")
    .getOrCreate()
)

# Read Files:

In [ ]:
base_dir = "../data"
plot_path = '../plots'

In [ ]:
final_path = base_dir + '/developed/final_data'
final_sdf = spark.read.parquet(final_path)
final_sdf.show(5)

In [ ]:
attributes = [
    "month", "day_type", "pickup_hour", "avg_trip_miles", "avg_utilization_rate", 
    "avg_total_fare_amount", "avg_total_revenue",
    "avg_bcf", "avg_trip_speed", "AvgHourlyTemp", "avg_hourly_demand"
            ]

final_df = final_sdf.select(attributes).toPandas()

In [ ]:
merge_path = base_dir + '/developed/merged_data/hourly_demand_hvfhv_weather_pluto'
merge_sdf = spark.read.parquet(merge_path)
merge_sdf.show(5)

# Draw Histograms of Numeric Attributes:

### No Transformation:

In [ ]:
import matplotlib.pyplot as plt
import os

# Set the figure size
plt.figure(figsize=(20, 20))

# Loop through each attribute and create a subplot
for i, attr in enumerate(attributes):
    plt.subplot(5, 4, i + 1)
    final_df[attr].hist(bins=30)
    plt.title(attr, weight='bold')  # Increased font size for visibility
    plt.xlabel(attr)
    plt.ylabel('Frequency', fontsize=12)

# Adjust layout with more spacing to prevent overlapping
plt.tight_layout()

# Save the plot to the specified file path
file_name = 'distribution_of_data.png'
file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

# Display the plot
plt.show()

### Apply log Transformation to Some Numerical Attributes:

In [ ]:
LOG_COLS = [
    "avg_trip_miles", "avg_total_fare_amount", "avg_bcf", 
    "avg_total_revenue","avg_hourly_demand"
            ]

# Ensure that no negative or zero values ​​appear in the logarithmic transformation
def log_transform(x):
    x = np.clip(x, a_min=0, a_max=None)
    return np.log1p(x)

# Calculate the data after logarithmic transformation
final_df_log = final_df.copy()

for attr in LOG_COLS:
    if attr in final_df.columns:
        final_df_log[attr] = log_transform(final_df[attr].fillna(0))

# Plot
plt.figure(figsize=(15, 15))

for i, attr in enumerate(attributes):
    if attr in LOG_COLS:
        plt.subplot(5, 4, i + 1)
        final_df_log[attr].hist(bins=30)
        plt.title(f"log {attr}", weight='bold')
        plt.xlabel(attr)
        plt.ylabel('Frequency')
    else:
        plt.subplot(5, 4, i + 1)
        final_df[attr].hist(bins=30)
        plt.title(attr, weight='bold')
        plt.xlabel(attr)
        plt.ylabel('Frequency')

plt.tight_layout()

# Save and display the plot
file_name = 'distribution_of_log_data.png'

file_path = os.path.join(plot_path, file_name)
plt.savefig(file_path, dpi=300, bbox_inches='tight')

plt.show()

# Apply log Transformations to Columns:

In [ ]:
# Extract unique values ​​for each column
pulocationid_labels = merge_sdf.select('PULocationID').distinct().rdd.flatMap(lambda x: x).collect()
service_zone_labels = merge_sdf.select('service_zone').distinct().rdd.flatMap(lambda x: x).collect()
zone_labels = merge_sdf.select('zone').distinct().rdd.flatMap(lambda x: x).collect()
borough_labels = merge_sdf.select('borough').distinct().rdd.flatMap(lambda x: x).collect()

# Transform labels to the desired format
pulocationid_labels = [f'PULocationID_{label}' for label in pulocationid_labels]
service_zone_labels = [f'service_zone_{label}' for label in service_zone_labels]
zone_labels = [f'zone_{label}' for label in zone_labels]
borough_labels = [f'borough_{label}' for label in borough_labels]

# Expand the `building_classes` column
exploded_df = merge_sdf.withColumn("building_class", explode(col("building_classes")))
# Get all unique tags
building_class_labels = exploded_df.select("building_class").distinct().rdd.flatMap(lambda x: x).collect()

# Put all unique values ​​in a list
unique_labels = list(set(pulocationid_labels + service_zone_labels + zone_labels + borough_labels + building_class_labels))

print(unique_labels)

In [ ]:
# Select the necessary columns
selected_final_sdf = final_sdf.select(attributes + unique_labels)

# Log transformation:
for col_name in LOG_COLS:
    selected_final_sdf = selected_final_sdf.withColumn(f'log_{col_name}', log(col(col_name) + 1))

# Drop original columns
columns_to_drop = LOG_COLS
selected_final_sdf = selected_final_sdf.drop(*columns_to_drop)

# Show the resulting DataFrame
selected_final_sdf.show(5)

In [ ]:
num_rows = selected_final_sdf.count()
print(f"Number of rows: {num_rows}")

columns = selected_final_sdf.columns
num_columns = len(columns)
print(f"Number of columns: {num_columns}")

# Save the Final Transformed Dataset:

In [ ]:
selected_final_sdf_dir = base_dir + '/developed'
file_name = 'selected_trans_final_data'
selected_final_sdf_path = os.path.join(selected_final_sdf_dir, file_name)
selected_final_sdf.write.mode('overwrite').parquet(selected_final_sdf_path)